# Tracking del pallone: YOLO addestrato + filtro di Kalman

Notebook separato: usa il checkpoint locale best.pt, senza training o download. YOLO OSS e la posizione grezza della detection; non viene sostituita da una coordinata Kalman ritardata. PRED e una stima del moto solo quando manca una detection.

Il filtro usa un modello cinematico ad accelerazione costante sul piano immagine (posizione, velocita e accelerazione in pixel), con rumore di processo di jerk. Non e un modello fisico 3D e non puo anticipare un urto improvviso non osservato: i tocchi si riflettono solo quando arriva una nuova detection, mentre un'innovazione incompatibile azzera lo stato cinematico mantenendo l'ID entro il timeout.

Per evitare predizioni che scappano, `MAX_PREDICTION_SECONDS` e `MAX_PREDICTION_STD_PX` limitano l'output PRED separatamente da `MAX_MISSED_SECONDS`, che conserva l'ID per la riacquisizione. Una predizione viene inoltre interrotta se esce dai bordi dell'immagine, senza clamp. Le detection continuano su ogni frame campionato e hanno sempre priorita.


In [11]:
from pathlib import Path
from datetime import datetime
import math

# Cerca la cartella di lavoro a partire dalla directory del notebook.
WORKSPACE_DIR = next(
    (p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "Annotazioni").is_dir() and (p / "runs" / "volleyball_ball").is_dir()),
    None,
)
if WORKSPACE_DIR is None:
    raise FileNotFoundError("Non trovo Annotazioni/ e runs/volleyball_ball/. Avvia il notebook dalla cartella di lavoro o da una sua sottocartella.")

VIDEO_DIR = WORKSPACE_DIR / "Annotazioni"
VIDEO_FILES = sorted(p for p in VIDEO_DIR.iterdir() if p.is_file() and p.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv", ".m4v"})
if not VIDEO_FILES:
    raise FileNotFoundError(f"Nessun video supportato in {VIDEO_DIR}")

# Solo best.pt locali appartenenti alle run del fine-tuning. Nessun fallback ai pesi generici.
CHECKPOINT_FILES = sorted(
    (WORKSPACE_DIR / "runs" / "volleyball_ball").glob("**/weights/best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not CHECKPOINT_FILES:
    raise FileNotFoundError(f"Non trovo checkpoint best.pt addestrati sotto {WORKSPACE_DIR / 'runs' / 'volleyball_ball'}. Esegui/completa prima il fine-tuning; non user? yolov8m.pt generico.")

print("Checkpoint addestrati trovati (indice, run, data file, percorso):")
for i, path in enumerate(CHECKPOINT_FILES, start=1):
    run_dir = path.parent.parent
    stamp = datetime.fromtimestamp(path.stat().st_mtime).astimezone().isoformat(timespec="seconds")
    print(f"{i}. {run_dir.name} | {stamp} | {path.relative_to(WORKSPACE_DIR)}")
print()
print("Video disponibili:")
for i, path in enumerate(VIDEO_FILES, start=1):
    print(f"{i}. {path.relative_to(WORKSPACE_DIR)} ({path.stat().st_size / (1024**3):.2f} GiB)")

# Se esiste un solo checkpoint lo seleziona; con pi? candidati impostare l'indice desiderato.
CHECKPOINT_INDEX = 1 if len(CHECKPOINT_FILES) == 1 else None
VIDEO_INDEX = 3

# Modificare CHECKPOINT_INDEX dopo aver esaminato la lista stampata se ci sono pi? run.
if CHECKPOINT_INDEX is None:
    raise ValueError(f"Sono presenti {len(CHECKPOINT_FILES)} checkpoint: imposta CHECKPOINT_INDEX esplicitamente tra 1 e {len(CHECKPOINT_FILES)}.")
if not 1 <= CHECKPOINT_INDEX <= len(CHECKPOINT_FILES):
    raise IndexError("CHECKPOINT_INDEX fuori intervallo.")
if not 1 <= VIDEO_INDEX <= len(VIDEO_FILES):
    raise IndexError("VIDEO_INDEX fuori intervallo.")
CHECKPOINT_PATH = CHECKPOINT_FILES[CHECKPOINT_INDEX - 1].resolve()
VIDEO_PATH = VIDEO_FILES[VIDEO_INDEX - 1].resolve()
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint selezionato non esiste: {CHECKPOINT_PATH}")

# Esempio breve, non l'intera partita. Il tempo ? in secondi del video.
START_SECONDS = 11*60+20
END_SECONDS = 2*3600+2*60+30
FRAME_STRIDE = 1  # inferenza a intervalli; dt KF conserva il tempo video reale.
IMAGE_SIZE = 640
BALL_CLASS_ID = 0  # verificare il nome classe stampato dopo il caricamento
CONFIDENCE_THRESHOLD = 0.77
NMS_IOU_THRESHOLD = 0.45
MAX_DETECTIONS = 20

# Modello CA in pixel: stato (x, y, vx, vy, ax, ay); tempi in secondi del video.
MAX_MISSED_SECONDS = 0.75  # timeout ID/riacquisizione, indipendente dall'orizzonte PRED.
MAX_PREDICTION_SECONDS = 0.15  # dopo questo orizzonte non si emettono PRED, ma l'ID resta fino al timeout.
MAX_PREDICTION_STD_PX = 30.0  # massimo autovalore std della covarianza XY, in pixel.
MEASUREMENT_STD_PX = 3.0  # R = std^2 I = 36 px^2 I; fisso, non dedotto dalla confidence e non nullo.
PROCESS_JERK_STD_PX_S3 = 3000.0  # rumore di processo CA (jerk bianco), px/s^3.
INITIAL_VELOCITY_STD_PX_S = 2200.0
INITIAL_ACCELERATION_STD_PX_S2 = 1200.0
ASSOCIATION_GATE_MAHALANOBIS_SQ = 25
REQUEST_GPU = True

if not math.isfinite(START_SECONDS) or not math.isfinite(END_SECONDS) or START_SECONDS < 0 or END_SECONDS <= START_SECONDS:
    raise ValueError("Configura un intervallo valido con 0 <= START_SECONDS < END_SECONDS.")
if isinstance(FRAME_STRIDE, bool) or not isinstance(FRAME_STRIDE, int) or FRAME_STRIDE < 1:
    raise ValueError("FRAME_STRIDE deve essere un intero positivo.")
if MAX_MISSED_SECONDS < 0 or not math.isfinite(MAX_MISSED_SECONDS):
    raise ValueError("MAX_MISSED_SECONDS deve essere finito e non negativo.")
if not math.isfinite(MAX_PREDICTION_SECONDS) or not 0 < MAX_PREDICTION_SECONDS <= MAX_MISSED_SECONDS:
    raise ValueError("MAX_PREDICTION_SECONDS deve essere positivo e non superare il timeout ID.")
if not math.isfinite(MAX_PREDICTION_STD_PX) or MAX_PREDICTION_STD_PX <= 0:
    raise ValueError("MAX_PREDICTION_STD_PX deve essere finito e positivo.")
if (not math.isfinite(MEASUREMENT_STD_PX) or MEASUREMENT_STD_PX <= 0
        or not math.isfinite(PROCESS_JERK_STD_PX_S3) or PROCESS_JERK_STD_PX_S3 < 0
        or not math.isfinite(INITIAL_VELOCITY_STD_PX_S) or INITIAL_VELOCITY_STD_PX_S < 0
        or not math.isfinite(INITIAL_ACCELERATION_STD_PX_S2) or INITIAL_ACCELERATION_STD_PX_S2 < 0):
    raise ValueError("Rumori KF non validi: misura >0 e deviazioni di processo/iniziali finite e >=0.")
if not math.isfinite(ASSOCIATION_GATE_MAHALANOBIS_SQ) or ASSOCIATION_GATE_MAHALANOBIS_SQ <= 0:
    raise ValueError("ASSOCIATION_GATE_MAHALANOBIS_SQ deve essere finito e positivo.")
if not 0 <= CONFIDENCE_THRESHOLD <= 1 or not 0 < NMS_IOU_THRESHOLD <= 1:
    raise ValueError("Soglie confidence/NMS fuori intervallo.")

print()
print(f"Checkpoint selezionato: {CHECKPOINT_PATH}")
print(f"Video: {VIDEO_PATH.name}; intervallo {START_SECONDS:g}?{END_SECONDS:g}s; stride {FRAME_STRIDE}")

Checkpoint addestrati trovati (indice, run, data file, percorso):
1. yolov8m_ball_20261114_172150 | 2026-11-14T19:03:34+01:00 | runs\volleyball_ball\yolov8m_ball_20261114_172150\weights\best.pt

Video disponibili:
1. Annotazioni\ITA_POL_VNL2024_30fps.mp4 (1.39 GiB)
2. Annotazioni\ITA_SRB_Parigi2024_30fps.mp4 (0.99 GiB)
3. Annotazioni\Test_demo_ITA_DOM_Parigi2024_30fps.mp4 (1.54 GiB)

Checkpoint selezionato: C:\Users\SavarDar\OneDrive - Electrolux\Desktop\Scout_Volley_CV&NLP\runs\volleyball_ball\yolov8m_ball_20261114_172150\weights\best.pt
Video: Test_demo_ITA_DOM_Parigi2024_30fps.mp4; intervallo 680?7350s; stride 1


## Dipendenze e preflight GPU/NMS

Usa le dipendenze gi? presenti nel kernel. Se mancano o le build PyTorch/torchvision sono incompatibili, non reinstallare pacchetti nel Python del training: crea un ambiente Jupyter isolato con versioni ufficiali compatibili con Python, driver e GPU installati, poi riavvia il kernel. Il preflight prova realmente `torchvision.ops.nms` su CPU e, se disponibile, CUDA. Solo l'incompatibilit? specifica dell'operatore NMS CUDA attiva il fallback CPU; altri errori vengono propagati.

In [12]:
import importlib.util
import sys

missing = [name for name in ("torch", "torchvision", "ultralytics", "cv2", "numpy") if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(f"Dipendenze mancanti nel kernel {sys.executable}: {missing}. Usa un ambiente isolato; non installare nel kernel del training in corso.")

import torch
import torchvision
import cv2
import numpy as np
from ultralytics import YOLO

CUDA_AVAILABLE = bool(REQUEST_GPU and torch.cuda.is_available())

def nms_smoke_test(device):
    boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 11., 11.]], dtype=torch.float32, device=device)
    scores = torch.tensor([0.9, 0.8], dtype=torch.float32, device=device)
    kept = torchvision.ops.nms(boxes, scores, 0.5).tolist()
    if kept != [0]:
        raise RuntimeError(f"Output NMS inatteso su {device}: {kept}")

print(f"Python: {sys.executable} ({sys.version.split()[0]})")
print(f"torch {torch.__version__}; torchvision {torchvision.__version__}; CUDA build {torch.version.cuda}; CUDA disponibile {CUDA_AVAILABLE}")
nms_smoke_test("cpu")
print("NMS CPU: OK")
if CUDA_AVAILABLE:
    try:
        nms_smoke_test("cuda:0")
    except NotImplementedError as exc:
        text = str(exc)
        if "torchvision::nms" not in text or "CUDA" not in text:
            raise
        DEVICE = "cpu"
        print("ATTENZIONE: torchvision NMS CUDA non disponibile; fallback CPU esplicito (inference pi? lenta).")
    else:
        DEVICE = 0
        print(f"NMS CUDA: OK ? {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    print("GPU CUDA non disponibile o non richiesta: fallback CPU esplicito.")

# Caricamento esclusivamente dal best.pt selezionato; non vengono scaricati pesi.
MODEL = YOLO(str(CHECKPOINT_PATH))
NAMES = MODEL.names
print("Classi checkpoint:", NAMES)
if isinstance(NAMES, dict):
    class_name = NAMES.get(BALL_CLASS_ID)
else:
    class_name = NAMES[BALL_CLASS_ID] if 0 <= BALL_CLASS_ID < len(NAMES) else None
if class_name is None:
    raise ValueError(f"BALL_CLASS_ID={BALL_CLASS_ID} non esiste nelle classi del checkpoint {NAMES}.")
print(f"Classe rilevata: id={BALL_CLASS_ID}, nome={class_name!r}; dispositivo inference: {DEVICE}")

Python: c:\Users\SavarDar\AppData\Local\Programs\Python\Python314\python.exe (3.14.6)
torch 2.14.1+cu130; torchvision 0.29.1+cu130; CUDA build 13.0; CUDA disponibile True
NMS CPU: OK
NMS CUDA: OK ? NVIDIA RTX PRO 2000 Blackwell Generation Laptop GPU
Classi checkpoint: {0: 'ball'}
Classe rilevata: id=0, nome='ball'; dispositivo inference: 0


## Modello CA e comportamento monopalla

`OSS` e sempre la posizione grezza YOLO; `PRED` e usata soltanto quando manca una detection. Lo stato Kalman e `(x, y, vx, vy, ax, ay)` in pixel. Il modello ad accelerazione costante si adatta a curvature graduali; un rumore di processo di jerk con unita `px/s^3` permette variazioni d'accelerazione coerenti con il `dt` video (quindi anche con lo stride). Non e un modello balistico o una ricostruzione 3D.

L'aggiornamento usa il guadagno Kalman standard. E stato rimosso `INNOVATION_VELOCITY_GAIN` (correzione alpha-beta ad hoc del precedente modello a velocita costante); il rumore di processo e passato da `ACCELERATION_STD_PX_S2` a `PROCESS_JERK_STD_PX_S3`, con `INITIAL_ACCELERATION_STD_PX_S2` per l'incertezza iniziale dell'accelerazione. `MEASUREMENT_STD_PX=6` conserva `R=36 px^2 I`: confidence non e interpretata come precisione spaziale.

Una detection forte e sempre emessa come unica `OSS`. Se incompatibile col gate, reinizializza posizione, velocita e accelerazione mantenendo l'ID finche non scade `MAX_MISSED_SECONDS`; e un reset cinematico, non riconoscimento del tocco. Senza detection si emette al massimo una `PRED` per frame, finche eta, incertezza e posizione restano entro i limiti. Superato un limite, le predizioni si fermano fino a una nuova OSS, ma lo stato/ID resta disponibile alla riacquisizione fino al timeout. Se la predizione esce dall'immagine non viene clampata al bordo.


In [13]:
from dataclasses import dataclass
import csv
import json

@dataclass
class BallTrack:
    track_id: int
    state: np.ndarray
    covariance: np.ndarray
    last_observed_time: float
    last_predict_time: float
    last_confidence: float
    prediction_stopped: bool = False

class SingleBallKalmanTracker:
    def __init__(self, max_missed_seconds, max_prediction_seconds, max_prediction_std_px,
                 measurement_std_px, process_jerk_std_px_s3, initial_velocity_std_px_s,
                 initial_acceleration_std_px_s2, gate_mahalanobis_sq):
        self.max_missed_seconds = float(max_missed_seconds)
        self.max_prediction_seconds = float(max_prediction_seconds)
        self.max_prediction_std_px = float(max_prediction_std_px)
        self.measurement_std_px = float(measurement_std_px)
        self.process_jerk_std_px_s3 = float(process_jerk_std_px_s3)
        self.initial_velocity_std_px_s = float(initial_velocity_std_px_s)
        self.initial_acceleration_std_px_s2 = float(initial_acceleration_std_px_s2)
        self.gate_mahalanobis_sq = float(gate_mahalanobis_sq)
        values = (self.max_missed_seconds, self.max_prediction_seconds, self.max_prediction_std_px,
                  self.measurement_std_px, self.process_jerk_std_px_s3,
                  self.initial_velocity_std_px_s, self.initial_acceleration_std_px_s2,
                  self.gate_mahalanobis_sq)
        if (not all(math.isfinite(value) for value in values)
                or self.max_missed_seconds < 0
                or not 0 < self.max_prediction_seconds <= self.max_missed_seconds
                or self.max_prediction_std_px <= 0 or self.measurement_std_px <= 0
                or self.process_jerk_std_px_s3 < 0 or self.initial_velocity_std_px_s < 0
                or self.initial_acceleration_std_px_s2 < 0 or self.gate_mahalanobis_sq <= 0):
            raise ValueError("Parametri CA Kalman, orizzonte o gate non validi.")
        self.track = None
        self.next_id = 1
        self.last_timestamp = None
        self.H = np.zeros((2, 6), dtype=np.float64)
        self.H[0, 0] = self.H[1, 1] = 1.0
        self.R = np.eye(2, dtype=np.float64) * self.measurement_std_px**2

    def _transition(self, dt):
        dt = float(dt)
        if not math.isfinite(dt) or dt < 0:
            raise ValueError("dt CA deve essere finito e non negativo.")
        F = np.eye(6, dtype=np.float64)
        F[0, 2], F[1, 3] = dt, dt
        F[0, 4], F[1, 5] = 0.5 * dt**2, 0.5 * dt**2
        F[2, 4], F[3, 5] = dt, dt
        G = np.zeros((6, 2), dtype=np.float64)
        G[[0, 2, 4], 0] = [dt**3 / 6.0, dt**2 / 2.0, dt]
        G[[1, 3, 5], 1] = [dt**3 / 6.0, dt**2 / 2.0, dt]
        Q = self.process_jerk_std_px_s3**2 * (G @ G.T)
        return F, Q

    def _predict(self, track, timestamp):
        dt = float(timestamp - track.last_predict_time)
        if not math.isfinite(dt) or dt < -1e-9:
            raise ValueError("Timestamp video non monotono.")
        if dt <= 0:
            return
        F, Q = self._transition(dt)
        track.state = F @ track.state
        track.covariance = F @ track.covariance @ F.T + Q
        track.covariance = 0.5 * (track.covariance + track.covariance.T)
        if not np.all(np.isfinite(track.state)) or not np.all(np.isfinite(track.covariance)):
            raise FloatingPointError("Stato/covarianza Kalman non finiti.")
        track.last_predict_time = float(timestamp)

    def _initial_covariance(self):
        return np.diag([
            self.measurement_std_px**2, self.measurement_std_px**2,
            self.initial_velocity_std_px_s**2, self.initial_velocity_std_px_s**2,
            self.initial_acceleration_std_px_s2**2, self.initial_acceleration_std_px_s2**2,
        ]).astype(np.float64)

    def _new_track(self, point, confidence, timestamp):
        track = BallTrack(
            track_id=self.next_id,
            state=np.array([point[0], point[1], 0., 0., 0., 0.], dtype=np.float64),
            covariance=self._initial_covariance(),
            last_observed_time=float(timestamp),
            last_predict_time=float(timestamp),
            last_confidence=float(confidence),
        )
        self.next_id += 1
        self.track = track
        return track

    def _distance_sq(self, track, point):
        innovation = np.asarray(point, dtype=np.float64) - self.H @ track.state
        S = self.H @ track.covariance @ self.H.T + self.R
        return float(innovation.T @ np.linalg.solve(S, innovation))

    def _update(self, track, point, confidence, timestamp):
        z = np.asarray(point, dtype=np.float64)
        innovation = z - self.H @ track.state
        S = self.H @ track.covariance @ self.H.T + self.R
        cross = track.covariance @ self.H.T
        K = np.linalg.solve(S, cross.T).T
        track.state = track.state + K @ innovation
        I_KH = np.eye(6, dtype=np.float64) - K @ self.H
        track.covariance = I_KH @ track.covariance @ I_KH.T + K @ self.R @ K.T
        track.covariance = 0.5 * (track.covariance + track.covariance.T)
        track.last_observed_time = float(timestamp)
        track.last_predict_time = float(timestamp)
        track.last_confidence = float(confidence)
        track.prediction_stopped = False

    def _reset(self, track, point, confidence, timestamp):
        track.state = np.array([point[0], point[1], 0., 0., 0., 0.], dtype=np.float64)
        track.covariance = self._initial_covariance()
        track.last_observed_time = float(timestamp)
        track.last_predict_time = float(timestamp)
        track.last_confidence = float(confidence)
        track.prediction_stopped = False

    def _position_std_px(self, track):
        position_covariance = track.covariance[:2, :2]
        if not np.all(np.isfinite(position_covariance)):
            return math.inf
        largest_eigenvalue = float(np.max(np.linalg.eigvalsh(position_covariance)))
        return math.sqrt(max(0.0, largest_eigenvalue))

    def _inside_image(self, track, image_shape):
        if image_shape is None:
            return True
        if len(image_shape) < 2:
            raise ValueError("image_shape deve contenere altezza e larghezza.")
        height, width = int(image_shape[0]), int(image_shape[1])
        if height <= 0 or width <= 0:
            raise ValueError("Dimensioni immagine non valide.")
        x, y = track.state[:2]
        return 0.0 <= x < width and 0.0 <= y < height

    def step(self, detections, timestamp, frame_index, image_shape=None):
        timestamp = float(timestamp)
        if not math.isfinite(timestamp) or (self.last_timestamp is not None and timestamp < self.last_timestamp - 1e-9):
            raise ValueError("Timestamp video non valido o non monotono.")
        self.last_timestamp = timestamp
        if self.track is not None and timestamp - self.track.last_observed_time > self.max_missed_seconds + 1e-9:
            self.track = None

        detections = select_single_detection(detections, CONFIDENCE_THRESHOLD)
        if not detections:
            if self.track is None:
                return [], {}
            track = self.track
            self._predict(track, timestamp)
            if not track.prediction_stopped:
                age = timestamp - track.last_observed_time
                safe = (age <= self.max_prediction_seconds + 1e-9
                        and self._position_std_px(track) <= self.max_prediction_std_px
                        and self._inside_image(track, image_shape))
                if not safe:
                    track.prediction_stopped = True
            if track.prediction_stopped:
                return [], {}
            row = {"timestamp_seconds": timestamp, "frame_index": int(frame_index), "track_id": track.track_id,
                   "confidence": None, "x_px": float(track.state[0]), "y_px": float(track.state[1]),
                   "position_type": "predetta", "bbox_x1": None, "bbox_y1": None, "bbox_x2": None, "bbox_y2": None}
            return [row], {}

        cx, cy, confidence, bbox = detections[0]
        point = np.array([cx, cy], dtype=np.float64)
        if self.track is None:
            track = self._new_track(point, confidence, timestamp)
        else:
            track = self.track
            self._predict(track, timestamp)
            if self._distance_sq(track, point) <= self.gate_mahalanobis_sq:
                self._update(track, point, confidence, timestamp)
            else:
                self._reset(track, point, confidence, timestamp)
        row = {"timestamp_seconds": timestamp, "frame_index": int(frame_index), "track_id": track.track_id,
               "confidence": float(confidence), "x_px": float(cx), "y_px": float(cy), "position_type": "osservata",
               "bbox_x1": float(bbox[0]), "bbox_y1": float(bbox[1]), "bbox_x2": float(bbox[2]), "bbox_y2": float(bbox[3])}
        return [row], {0: track.track_id}

def select_single_detection(detections, confidence_threshold):
    """Keep the highest-confidence valid candidate; stable ordering breaks ties."""
    valid = [d for d in detections if math.isfinite(float(d[2])) and float(d[2]) >= confidence_threshold]
    return [max(valid, key=lambda detection: float(detection[2]))] if valid else []

TRACKER_PARAMETERS = dict(
    max_missed_seconds=MAX_MISSED_SECONDS,
    max_prediction_seconds=MAX_PREDICTION_SECONDS,
    max_prediction_std_px=MAX_PREDICTION_STD_PX,
    measurement_std_px=MEASUREMENT_STD_PX,
    process_jerk_std_px_s3=PROCESS_JERK_STD_PX_S3,
    initial_velocity_std_px_s=INITIAL_VELOCITY_STD_PX_S,
    initial_acceleration_std_px_s2=INITIAL_ACCELERATION_STD_PX_S2,
    gate_mahalanobis_sq=ASSOCIATION_GATE_MAHALANOBIS_SQ,
)
print("Tracker monopalla CA e limiti di predizione pronti.")


Tracker monopalla CA e limiti di predizione pronti.


## Test sintetici: curvatura, tocchi e limiti PRED

I test confrontano il modello CA con un riferimento che replica il filtro precedente CV + guadagno innovazione, su piu traiettorie a accelerazione costante. Verificano inoltre cambio brusco, reset su innovazione incompatibile, moto uniforme/occlusione, timeout/ID, dt/stride, stop per orizzonte, incertezza e bordo, e covarianze/Q finite e semidefinite positive. Sono test sintetici, non metriche di accuratezza su video.


In [14]:
# Test deterministici sintetici. Nessun output su video viene scritto da questa cella.
TEST_DT = 2.0 / 30.0
TEST_CONFIDENCE = max(0.95, CONFIDENCE_THRESHOLD)

def _test_detection(x, y=200.0, confidence=TEST_CONFIDENCE):
    return (float(x), float(y), float(confidence), (float(x-1), float(y-1), float(x+1), float(y+1)))

def _new_test_tracker(**overrides):
    params = dict(TRACKER_PARAMETERS)
    params.update(overrides)
    return SingleBallKalmanTracker(**params)

def _curve_position(t, ax, ay):
    return np.array([100.0 + 180.0*t + 0.5*ax*t*t,
                     300.0 + 50.0*t + 0.5*ay*t*t], dtype=np.float64)

class _PreviousCVReference:
    """Riferimento sintetico del precedente CV con R corrente e correzione alpha-beta."""
    def __init__(self):
        self.state = None
        self.covariance = None
        self.last_time = None
        self.H = np.array([[1.,0.,0.,0.],[0.,1.,0.,0.]])
        self.R = np.eye(2) * MEASUREMENT_STD_PX**2

    def step(self, point, timestamp):
        if self.state is None:
            self.state = np.array([point[0], point[1], 0., 0.], dtype=np.float64)
            self.covariance = np.diag([MEASUREMENT_STD_PX**2, MEASUREMENT_STD_PX**2,
                                       INITIAL_VELOCITY_STD_PX_S**2, INITIAL_VELOCITY_STD_PX_S**2])
            self.last_time = timestamp
            return None
        dt = timestamp - self.last_time
        F = np.array([[1.,0.,dt,0.],[0.,1.,0.,dt],[0.,0.,1.,0.],[0.,0.,0.,1.]])
        G = np.array([[0.5*dt*dt,0.],[0.,0.5*dt*dt],[dt,0.],[0.,dt]])
        Q = 1050.0**2 * (G @ G.T)
        predicted = F @ self.state
        predicted_covariance = F @ self.covariance @ F.T + Q
        error = float(np.linalg.norm(point - predicted[:2]))
        innovation = point - self.H @ predicted
        S = self.H @ predicted_covariance @ self.H.T + self.R
        distance_sq = float(innovation.T @ np.linalg.solve(S, innovation))
        if distance_sq <= ASSOCIATION_GATE_MAHALANOBIS_SQ:
            K = predicted_covariance @ self.H.T @ np.linalg.inv(S)
            K[2:, :] += (0.50 / dt) * np.eye(2)
            self.state = predicted + K @ innovation
            I_KH = np.eye(4) - K @ self.H
            self.covariance = I_KH @ predicted_covariance @ I_KH.T + K @ self.R @ K.T
        else:
            self.state = np.array([point[0], point[1], 0., 0.], dtype=np.float64)
            self.covariance = np.diag([MEASUREMENT_STD_PX**2, MEASUREMENT_STD_PX**2,
                                       INITIAL_VELOCITY_STD_PX_S**2, INITIAL_VELOCITY_STD_PX_S**2])
        self.last_time = timestamp
        return error

def _curve_rmse(ax, ay, use_ca):
    ca = _new_test_tracker() if use_ca else None
    old = _PreviousCVReference() if not use_ca else None
    errors = []
    for sample_index in range(41):
        timestamp = sample_index * TEST_DT
        point = _curve_position(timestamp, ax, ay)
        if use_ca:
            if ca.track is not None:
                dt = timestamp - ca.track.last_predict_time
                F, _ = ca._transition(dt)
                errors.append(float(np.linalg.norm(point - (F @ ca.track.state)[:2])))
            rows, _ = ca.step([_test_detection(*point)], timestamp, sample_index*2, (1080,1920))
            assert len(rows) == 1 and rows[0]["position_type"] == "osservata"
        else:
            error = old.step(point, timestamp)
            if error is not None:
                errors.append(error)
    return float(np.sqrt(np.mean(np.square(errors))))

curve_results = []
for acceleration in ((180.0, 80.0), (320.0, 140.0)):
    previous_rmse = _curve_rmse(*acceleration, use_ca=False)
    ca_rmse = _curve_rmse(*acceleration, use_ca=True)
    curve_results.append((previous_rmse, ca_rmse))
    assert ca_rmse < previous_rmse, (acceleration, previous_rmse, ca_rmse)
print("RMSE previsione one-step su due traiettorie curve sintetiche:",
      [(round(old,2), round(new,2)) for old,new in curve_results], "(CV precedente, CA nuovo; px)")

# Brusca inversione osservata: il filtro deve apprendere il verso opposto senza perdere l'ID.
turn = _new_test_tracker()
turn_id = None
for sample_index in range(31):
    timestamp = sample_index * TEST_DT
    x = 100.0 + 260.0*timestamp if timestamp <= 0.6 else 256.0 - 180.0*(timestamp-0.6)
    rows, _ = turn.step([_test_detection(x)], timestamp, sample_index*2, (1080,1920))
    assert len(rows) == 1 and rows[0]["position_type"] == "osservata"
    turn_id = rows[0]["track_id"] if turn_id is None else turn_id
    assert rows[0]["track_id"] == turn_id
assert turn.track.state[2] < 0, f"La velocita non ha seguito l'inversione: vx={turn.track.state[2]}"

# Reset gate controllato: una misura incompatibile azzera velocita/accelerazione, senza PRED ghost.
reset = _new_test_tracker(process_jerk_std_px_s3=0.0, initial_velocity_std_px_s=0.0,
                          initial_acceleration_std_px_s2=0.0)
first, _ = reset.step([_test_detection(100, 100)], 0.0, 0, (500,500))
reset.track.state = np.array([100.,100.,800.,0.,1000.,0.])
reset.track.covariance = np.eye(6) * 4.0
changed, assignments = reset.step([_test_detection(400, 100)], TEST_DT, 2, (500,500))
assert len(changed) == 1 and changed[0]["position_type"] == "osservata"
assert changed[0]["track_id"] == first[0]["track_id"] and assignments == {0: first[0]["track_id"]}
assert np.array_equal(reset.track.state, np.array([400.,100.,0.,0.,0.,0.]))

# Moto uniforme e occlusione breve; una PRED sola, con timestamp/stride effettivi.
steady = _new_test_tracker()
for sample_index in range(3):
    timestamp = sample_index * TEST_DT
    rows, _ = steady.step([_test_detection(50.0+300.0*timestamp)], timestamp, sample_index*2, (1080,1920))
assert len(rows) == 1 and rows[0]["position_type"] == "osservata"
rows, _ = steady.step([], 4*TEST_DT, 10, (1080,1920))
assert len(rows) == 1 and rows[0]["position_type"] == "predetta" and rows[0]["confidence"] is None
steady_error = abs(rows[0]["x_px"] - (50.0+300.0*4*TEST_DT))
assert steady_error < 8.0, steady_error

# Detection monopalla: selezione deterministica della confidence massima.
selected = select_single_detection([_test_detection(10, confidence=0.81), _test_detection(90, confidence=0.98)], CONFIDENCE_THRESHOLD)
assert len(selected) == 1 and selected[0][0] == 90.0
single = _new_test_tracker()
rows, _ = single.step([_test_detection(10), _test_detection(90, confidence=0.98)], 0.0, 0, (1080,1920))
assert len(rows) == 1 and rows[0]["x_px"] == 90.0

# Limite orizzonte: smette PRED ma trattiene l'ID; scadenza separata e ritorno con ID nuovo.
horizon = _new_test_tracker()
for i in range(3):
    t = i*TEST_DT
    horizon.step([_test_detection(100.0+20.0*i)], t, i*2, (1080,1920))
old_id = horizon.track.track_id
within, _ = horizon.step([], 3*TEST_DT, 6, (1080,1920))
after_horizon, _ = horizon.step([], 6*TEST_DT, 12, (1080,1920))
assert len(within) == 1 and within[0]["position_type"] == "predetta"
assert after_horizon == [] and horizon.track is not None and horizon.track.track_id == old_id
expired, _ = horizon.step([], 0.90, 27, (1080,1920))
assert expired == [] and horizon.track is None
returned, _ = horizon.step([_test_detection(120)], 0.95, 28, (1080,1920))
assert len(returned) == 1 and returned[0]["track_id"] != old_id

# Stop per incertezza, anche prima dell'orizzonte; lo stato resta disponibile per riacquisizione.
uncertain = _new_test_tracker(max_prediction_std_px=8.0)
for i in range(3):
    uncertain.step([_test_detection(100.0+10.0*i)], i*TEST_DT, i*2, (1080,1920))
uncertain_rows, _ = uncertain.step([], 3*TEST_DT, 6, (1080,1920))
assert uncertain_rows == [] and uncertain.track is not None and uncertain.track.prediction_stopped

# Stop al bordo: non clampare e non disegnare; validazione posizione usa dimensioni immagine reali.
border = _new_test_tracker()
first, _ = border.step([_test_detection(95,50)], 0.0, 0, (100,100))
border.track.state = np.array([95.,50.,1000.,0.,0.,0.])
border.track.covariance = np.eye(6)
outside, _ = border.step([], TEST_DT, 2, (100,100))
assert outside == [] and border.track.prediction_stopped

# Il delta frame stride usa il tempo video reale, non il tempo di elaborazione.
stride = _new_test_tracker()
stride.step([_test_detection(100,100)], 0.0, 0, (1080,1920))
stride.step([_test_detection(100+300*(4/30),100)], 4/30, 4, (1080,1920))
assert abs(stride.track.last_predict_time - 4/30) < 1e-12

# Q e covarianze devono essere finite e semidefinite positive dopo predizione/aggiornamenti.
psd = _new_test_tracker()
for dt in (1/30, TEST_DT, 0.2):
    _, Q = psd._transition(dt)
    assert np.all(np.isfinite(Q)) and np.linalg.eigvalsh(Q).min() >= -1e-8
for i in range(20):
    t = i*TEST_DT
    psd.step([_test_detection(200+100*t, 300+40*t)], t, i*2, (1080,1920))
    if i % 3 == 2:
        psd.step([], t+TEST_DT, i*2+1, (1080,1920))
    eig = np.linalg.eigvalsh(psd.track.covariance)
    assert np.all(np.isfinite(eig)) and eig.min() >= -1e-7, eig
print(f"Cambio direzione/reset, moto uniforme+occlusione ({steady_error:.2f}px), monopalla, ID/timeout, orizzonte, incertezza, bordo, dt/stride e covarianze: OK.")


RMSE previsione one-step su due traiettorie curve sintetiche: [(2.39, 2.05), (2.62, 2.13)] (CV precedente, CA nuovo; px)
Cambio direzione/reset, moto uniforme+occlusione (0.06px), monopalla, ID/timeout, orizzonte, incertezza, bordo, dt/stride e covarianze: OK.


## Esecuzione su un estratto e salvataggio

Il writer prova MP4V e poi MJPG; dopo la scrittura riapre il file e verifica che sia leggibile. Timestamp e `dt` derivano dall'indice frame effettivo diviso per l'FPS sorgente. Con stride > 1 il KF usa il tempo effettivo tra i frame campionati e il video di output usa FPS `sorgente/stride`. La tabella contiene al massimo una riga per frame: `osservata` e la detection grezza; `predetta` e la stima CA solo quando manca la detection e i limiti di orizzonte, incertezza e immagine sono rispettati. Dopo lo stop PRED non ci sono righe finche non arriva una detection.

Gli output gia salvati sotto la cella di esecuzione sono storici e derivano dal filtro precedente; vengono conservati e non rappresentano questa nuova logica. Rieseguire la cella crea una nuova cartella timestampata e non sovrascrive le run esistenti.


In [15]:
OUTPUT_ROOT = WORKSPACE_DIR / "runs" / "ball_tracking"
OUTPUT_DIR = OUTPUT_ROOT / f"{VIDEO_PATH.stem}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
CSV_PATH = OUTPUT_DIR / "tracce.csv"
CONFIG_PATH = OUTPUT_DIR / "configurazione.json"
PREVIEW_PATH = OUTPUT_DIR / "preview_tracking_pallone.png"

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise OSError(f"Impossibile aprire il video: {VIDEO_PATH}")
source_fps = float(cap.get(cv2.CAP_PROP_FPS))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if not math.isfinite(source_fps) or source_fps <= 0 or frame_count <= 0:
    cap.release()
    raise ValueError(f"Metadati video non validi: fps={source_fps}, frame_count={frame_count}")
if END_SECONDS > frame_count / source_fps + 1.0 / source_fps:
    cap.release()
    raise ValueError(f"END_SECONDS supera la durata del video ({frame_count/source_fps:.2f}s).")
start_frame = int(round(START_SECONDS * source_fps))
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
tracker = SingleBallKalmanTracker(**TRACKER_PARAMETERS)
rows = []
writer = None
annotated_path = None
processed_frames = 0
next_frame_index = start_frame

try:
    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, next_frame_index)
        ok, image = cap.read()
        if not ok:
            break
        frame_index = int(round(cap.get(cv2.CAP_PROP_POS_FRAMES))) - 1
        if frame_index < 0:
            raise RuntimeError("OpenCV non ha fornito un indice frame valido.")
        timestamp = frame_index / source_fps
        if timestamp > END_SECONDS + 0.5 / source_fps:
            break
        if timestamp < START_SECONDS - 0.5 / source_fps:
            next_frame_index = frame_index + FRAME_STRIDE
            continue

        # Detection indipendente dallo stato del tracker: chiamata su ogni frame elaborato.
        prediction = MODEL.predict(source=image, imgsz=IMAGE_SIZE, conf=CONFIDENCE_THRESHOLD,
            iou=NMS_IOU_THRESHOLD, max_det=MAX_DETECTIONS, classes=[BALL_CLASS_ID],
            device=DEVICE, verbose=False)[0]
        detections = []
        if prediction.boxes is not None and len(prediction.boxes):
            xyxy = prediction.boxes.xyxy.detach().cpu().numpy()
            confs = prediction.boxes.conf.detach().cpu().numpy()
            for box, confidence in zip(xyxy, confs):
                x1, y1, x2, y2 = map(float, box)
                detections.append(((x1+x2)/2, (y1+y2)/2, float(confidence), (x1, y1, x2, y2)))

        detections = select_single_detection(detections, CONFIDENCE_THRESHOLD)
        frame_rows, assignments = tracker.step(detections, timestamp, frame_index, image.shape[:2])
        if len(frame_rows) > 1:
            raise RuntimeError("Il tracker monopalla ha prodotto piu righe sullo stesso frame.")
        rows.extend(frame_rows)
        rendered = image.copy()
        for det_index, (cx, cy, confidence, box) in enumerate(detections):
            track_id = assignments[det_index]
            x1, y1, x2, y2 = map(int, np.rint(box))
            cv2.rectangle(rendered, (x1, y1), (x2, y2), (0, 220, 0), 2)
            cv2.circle(rendered, (int(round(cx)), int(round(cy))), 5, (0, 255, 0), -1)
            cv2.putText(rendered, f"ID {track_id} OSS {confidence:.2f}", (x1, max(18, y1-7)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 0), 2, cv2.LINE_AA)
        for row in frame_rows:
            if row["position_type"] != "predetta":
                continue
            x, y = int(round(row["x_px"])), int(round(row["y_px"]))
            color = (0, 165, 255)
            cv2.line(rendered, (x-8, y), (x-3, y), color, 2)
            cv2.line(rendered, (x+3, y), (x+8, y), color, 2)
            cv2.line(rendered, (x, y-8), (x, y-3), color, 2)
            cv2.line(rendered, (x, y+3), (x, y+8), color, 2)
            cv2.putText(rendered, f"ID {row['track_id']} PRED", (x+9, y-7), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)
        cv2.rectangle(rendered, (0, 0), (390, 42), (0, 0, 0), -1)
        cv2.putText(rendered, f"{timestamp:.3f}s | detections {len(detections)}", (12, 29), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 1, cv2.LINE_AA)

        if writer is None:
            height, width = rendered.shape[:2]
            output_fps = source_fps / FRAME_STRIDE
            for fourcc_text, suffix in (("mp4v", ".mp4"), ("MJPG", ".avi")):
                candidate = OUTPUT_DIR / f"tracking{suffix}"
                attempt = cv2.VideoWriter(str(candidate), cv2.VideoWriter_fourcc(*fourcc_text), output_fps, (width, height))
                if attempt.isOpened():
                    writer, annotated_path = attempt, candidate
                    break
                attempt.release()
            if writer is None:
                raise RuntimeError("Codec MP4V e MJPG non sono disponibili per scrivere il video annotato.")
        writer.write(rendered)
        if processed_frames == 0:
            if not cv2.imwrite(str(PREVIEW_PATH), rendered):
                raise OSError(f"Impossibile salvare la preview: {PREVIEW_PATH}")
        processed_frames += 1
        if processed_frames % 10 == 0:
            print(f"Elaborati {processed_frames} frame; video t={timestamp:.2f}s; detection={len(detections)}")
        next_frame_index = frame_index + FRAME_STRIDE
finally:
    cap.release()
    if writer is not None:
        writer.release()

if processed_frames == 0 or annotated_path is None:
    raise RuntimeError("L'intervallo non ha prodotto frame annotati; controlla video/timestamp.")

columns = ["timestamp_seconds", "frame_index", "track_id", "confidence", "x_px", "y_px", "position_type", "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]
with CSV_PATH.open("w", newline="", encoding="utf-8-sig") as f:
    output = csv.DictWriter(f, fieldnames=columns)
    output.writeheader()
    output.writerows(rows)
config = {
    "video": str(VIDEO_PATH), "checkpoint": str(CHECKPOINT_PATH), "checkpoint_run": CHECKPOINT_PATH.parent.parent.name,
    "checkpoint_mtime": datetime.fromtimestamp(CHECKPOINT_PATH.stat().st_mtime).astimezone().isoformat(),
    "start_seconds": START_SECONDS, "end_seconds": END_SECONDS, "source_fps": source_fps,
    "frame_stride": FRAME_STRIDE, "output_fps": source_fps/FRAME_STRIDE, "processed_frames": processed_frames,
    "class_id": BALL_CLASS_ID, "class_name": str(class_name), "confidence_threshold": CONFIDENCE_THRESHOLD,
    "nms_iou_threshold": NMS_IOU_THRESHOLD, "device": str(DEVICE), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "tracker": TRACKER_PARAMETERS, "measurement_covariance_R_px2": float(MEASUREMENT_STD_PX**2),
    "single_ball_policy": "highest confidence candidate per sampled frame; incompatible detection resets CA state while retaining ID until timeout",
    "prediction_stop_policy": "stop PRED after configured horizon, position standard-deviation limit, or leaving image bounds; retain ID until MAX_MISSED_SECONDS; never clamp",
    "note": "OSS is raw YOLO measurement; PRED is CA-only during missing detections. No hidden-touch detection or 3D ball reconstruction."
}
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
# Verifica concreta che il codec/output scritto sia riapribile.
check = cv2.VideoCapture(str(annotated_path))
if not check.isOpened():
    raise OSError(f"Video annotato non riapribile: {annotated_path}")
readable, _ = check.read()
check.release()
if not readable:
    raise OSError(f"Il video annotato non contiene frame leggibili: {annotated_path}")
print(f"Frame elaborati: {processed_frames}; righe tracce: {len(rows)}")
print(f"Video annotato: {annotated_path}")
print(f"CSV: {CSV_PATH}")
print(f"Configurazione: {CONFIG_PATH}")
print(f"Preview: {PREVIEW_PATH}")
print("Tipo righe:", {kind: sum(r["position_type"] == kind for r in rows) for kind in ("osservata", "predetta")})

Elaborati 10 frame; video t=680.30s; detection=0
Elaborati 20 frame; video t=680.63s; detection=0
Elaborati 30 frame; video t=680.97s; detection=0
Elaborati 40 frame; video t=681.30s; detection=0
Elaborati 50 frame; video t=681.63s; detection=0
Elaborati 60 frame; video t=681.97s; detection=0
Elaborati 70 frame; video t=682.30s; detection=0
Elaborati 80 frame; video t=682.63s; detection=0
Elaborati 90 frame; video t=682.97s; detection=0
Elaborati 100 frame; video t=683.30s; detection=0
Elaborati 110 frame; video t=683.63s; detection=0
Elaborati 120 frame; video t=683.97s; detection=0
Elaborati 130 frame; video t=684.30s; detection=0
Elaborati 140 frame; video t=684.63s; detection=0
Elaborati 150 frame; video t=684.97s; detection=0
Elaborati 160 frame; video t=685.30s; detection=0
Elaborati 170 frame; video t=685.63s; detection=0
Elaborati 180 frame; video t=685.97s; detection=0
Elaborati 190 frame; video t=686.30s; detection=1
Elaborati 200 frame; video t=686.63s; detection=0
Elaborati

## Limiti interpretativi

Il modello CA in coordinate immagine puo seguire meglio curvature graduali, ma non predice la fisica di un urto improvviso e non riconosce contatti nascosti: senza nuove detection non puo sapere che il moto e cambiato. La nuova OSS resta prioritaria; se incompatibile col gate, la posizione viene accettata e lo stato di velocita/accelerazione e azzerato, mantenendo l'ID solo entro `MAX_MISSED_SECONDS`. Il modello non interpreta la confidence come varianza spaziale.

`MAX_PREDICTION_SECONDS`, `MAX_PREDICTION_STD_PX` e il bordo immagine interrompono l'emissione PRED senza spostare artificialmente il punto; il timeout ID e distinto. Il limite immagine non equivale ai confini del campo: non si applica un'omografia/ROI pavimento al pallone in volo. In riscaldamento con piu palloni il tracker monopalla puo saltare fra detections; non fa re-identificazione. Coordinate in pixel, non traiettoria 3D. I test sintetici non sono accuratezza misurata su video; rivedere overlay e CSV prima di usare i risultati come ground truth.
